In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import sys
import shutil
import importlib
from dataclasses import fields
import json
from datetime import datetime

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt


# Notebook location:
# THESIS/z.parcels_postprocessing/notebooks/run_particles_mr_sm.ipynb
NB_DIR = Path.cwd().resolve()
PROJECT_DIR = NB_DIR.parent.parent

# Add the project root and the two postprocessing folders.
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
if str(PROJECT_DIR / "z.flow_postprocessing") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "z.flow_postprocessing"))
if str(PROJECT_DIR / "z.parcels_postprocessing") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "z.parcels_postprocessing"))


# Project modules
import scripts.fieldset as pfs
import scripts.particles as pparticles
import scripts.kernels_common as kcommon
import scripts.kernels_passive as kpassive
import scripts.kernels_mr_sm as kmrsm
import scripts.run as prun
import scripts.notebook_helpers as phelp
import theme.plot_theme as ptheme


for module in [
    pfs,
    pparticles,
    kcommon,
    kpassive,
    kmrsm,
    prun,
    phelp,
    ptheme,
]:
    importlib.reload(module)

ptheme.apply_theme()

print(f"Notebook dir : {NB_DIR}")
print(f"Project dir  : {PROJECT_DIR}")
print(f"Parcels dir  : {PROJECT_DIR / 'z.parcels_postprocessing'}")
print(f"run.py loaded from: {Path(prun.__file__).resolve()}")
print(f"passive kernel loaded from: {Path(kpassive.__file__).resolve()}")
print(f"MR-SM kernel loaded from: {Path(kmrsm.__file__).resolve()}")


In [ ]:
# ============================================================
# 2. DIRECTORY / CASE SETTINGS
# ============================================================

case_name = "parcels_sea_win_spinupsim_L1"   # edit only this
RUN_TITLE_INFO = case_name  # keep compact; used only for metadata

INPUT_DIR = (NB_DIR / "../data/input").resolve()
DATA_FILE = (INPUT_DIR / f"{case_name}.nc").resolve()

if not DATA_FILE.exists():
    available = sorted(p.name for p in INPUT_DIR.glob("*"))
    raise FileNotFoundError(
        f"Could not find input file:\n{DATA_FILE}\n\n"
        f"Available files in {INPUT_DIR}:\n"
        + "\n".join(available)
    )


# ============================================================
# 3. PARCELS ADVECTION SETTINGS
# ============================================================

DAY_PER_INDEX = 3.0 / 24.0
TIME_STEP_SECONDS = int(DAY_PER_INDEX * 86400)

RELEASE_TIME_INDEX = 0
LEVEL_INDICES = (0,)

RUNTIME_MODE = "manual"       # "manual" or "full_forcing"
RUNTIME_DAYS_REQUESTED = 7.0

DT_SECONDS = 300
OUTPUTDT_SECONDS = TIME_STEP_SECONDS

NX = 60
NY = 60

PERIODIC = True
RELEASE_MARGIN_CELLS = 1.0

RUN_ADVECTION = True
OVERWRITE_OUTPUT = True

# For a separate analysis notebook, keep this True.
SAVE_TRAJECTORIES = True
SAVE_METADATA = True


# ============================================================
# 4. PARTICLE CLASS SETTINGS
# ============================================================

# Use a constant f-plane value for the flat-grid particle model.
# For your GPGP-like domain, choose a representative latitude manually.
OMEGA_EARTH = 7.2921e-5
REPRESENTATIVE_LAT_DEG = 32.5
F0 = 2.0 * OMEGA_EARTH * np.sin(np.deg2rad(REPRESENTATIVE_LAT_DEG))

# Reference timescale used to label/store St = tau_eff / T_ref.
# This does not directly enter the kernel.
FLOW_TIMESCALE_SECONDS = 20.0 * 86400.0

# Define the particle classes you want to run.
# Passive is added separately with include_passive=True.
TAU_P_SECONDS_LIST = [
    1.0 * 3600.0,
    3.0 * 3600.0,
    6.0 * 3600.0,
    12.0 * 3600.0,
]

particle_specs = phelp.build_particle_specs(
    tau_p_seconds_list=TAU_P_SECONDS_LIST,
    include_passive=True,
    flow_timescale_seconds=FLOW_TIMESCALE_SECONDS,
)

phelp.print_particle_specs(particle_specs)


# ============================================================
# 5. OUTPUT ORGANIZATION
# results/<case_name>/...
# ============================================================

release_time_days = RELEASE_TIME_INDEX * DAY_PER_INDEX
level_tag = "k" + "-".join(str(k) for k in LEVEL_INDICES)

has_mr_sm = any(spec["particle_class"] == "mr_sm" for spec in particle_specs)
particle_collection_tag = "passive_plus_mrsm" if has_mr_sm else "passive"
run_collection_id = f"{particle_collection_tag}_{level_tag}_release_t{RELEASE_TIME_INDEX:04d}"

paths = phelp.prepare_output_paths(
    notebook_dir=NB_DIR,
    case_name=case_name,
    run_collection_id=run_collection_id,
    save_trajectories=SAVE_TRAJECTORIES,
    save_metadata=SAVE_METADATA,
    save_figures=False,
    compute_statistics=False,
)

CASE_RESULTS_DIR = paths.case_results_dir
RUN_OUT_DIR = paths.run_out_dir
METADATA_DIR = paths.metadata_dir
CONFIG_JSON = paths.config_json

phelp.print_output_paths(
    DATA_FILE,
    INPUT_DIR,
    paths,
    save_trajectories=SAVE_TRAJECTORIES,
    save_figures=False,
)

print(f"\nRun collection ID: {run_collection_id}")
print(f"Collection config: {CONFIG_JSON}")


In [ ]:
# ============================================================
# 6. CHECK RUNCONFIG AND INPUT FIELD
# ============================================================

required_runconfig_fields = {
    "input_nc",
    "output_path",
    "runtime_days",
    "dt_seconds",
    "outputdt_seconds",
    "time_step_seconds",
    "release_time_index",
    "periodic",
    "level_indices",
    "release_margin_cells",
    "particle_class",
    "particle_tag",
    "particle_label",
    "tau_p_seconds",
    "f0",
    "flow_timescale_seconds",
}

available_runconfig_fields = {f.name for f in fields(prun.RunConfig)}
missing = required_runconfig_fields - available_runconfig_fields

if missing:
    raise RuntimeError(
        "RunConfig is missing required tau-only fields:\n"
        f"{sorted(missing)}\n\n"
        "Update scripts/run.py with tau_p_seconds, then restart the kernel or reload prun."
    )

# Optional warning only: these old fields should no longer be used in the notebook.
deprecated_fields = {
    "B",
    "diameter_m",
    "nu_m2_s",
    "drag_correction",
    "C_Rep",
    "Rep_max",
}

still_present = deprecated_fields & available_runconfig_fields
if still_present:
    print(
        "Note: old diameter/buoyancy/drag fields still exist in RunConfig, "
        "but this notebook will not use them:"
    )
    print(f"  {sorted(still_present)}")


raw_qc = pfs.summarize_dataset(DATA_FILE)

print("\nRaw MITgcm input QC")
print(f"  dims      : {raw_qc.get('dims')}")
print(f"  U dims    : {raw_qc.get('u_dims', 'not reported')}")
print(f"  V dims    : {raw_qc.get('v_dims', 'not reported')}")
print(f"  x center  : {raw_qc.get('x_center', raw_qc.get('x_name', 'unknown'))}")
print(f"  y center  : {raw_qc.get('y_center', raw_qc.get('y_name', 'unknown'))}")
print(f"  U var     : {raw_qc.get('u_name')}")
print(f"  V var     : {raw_qc.get('v_name')}")


# Build a lightweight fieldset for QC/runtime selection.
# No derivative fields are needed for this QC check.
# Derivatives are built inside prun.run_parcels_experiment only for MR-SM runs.
fieldset_qc, meta, ds_parcels = pfs.build_fieldset(
    DATA_FILE,
    surface_only=True,
    mesh="flat",
    level_indices=LEVEL_INDICES,
    time_step_seconds=TIME_STEP_SECONDS,
    periodic=PERIODIC,
    add_derivatives=False,
)

qc = pfs.quick_qc_parcels_input(ds_parcels)

print("\nParcels-ready field QC")
print(f"  dims      : {qc.get('dims')}")
print(f"  U shape   : {qc.get('U_shape', qc.get('u_shape'))}")
print(f"  V shape   : {qc.get('V_shape', qc.get('v_shape'))}")
print(f"  x range   : {qc.get('x_min'):.1f} to {qc.get('x_max'):.1f} m")
print(f"  y range   : {qc.get('y_min'):.1f} to {qc.get('y_max'):.1f} m")
print(f"  n time    : {qc.get('n_time', ds_parcels.sizes['time'])}")
print(f"  dt head   : {np.diff(ds_parcels['time'].values[:5])}")
print(f"  U NaN t0  : {qc.get('U_nan_fraction_t0', qc.get('u_nan_fraction_t0'))}")
print(f"  V NaN t0  : {qc.get('V_nan_fraction_t0', qc.get('v_nan_fraction_t0'))}")


# ============================================================
# 7. RUNTIME SELECTION
# ============================================================

n_forcing_times = ds_parcels.sizes["time"]

max_runtime_days = (
    (n_forcing_times - 1 - RELEASE_TIME_INDEX)
    * TIME_STEP_SECONDS
    / 86400.0
)

if RELEASE_TIME_INDEX < 0 or RELEASE_TIME_INDEX >= n_forcing_times:
    raise ValueError(
        f"RELEASE_TIME_INDEX={RELEASE_TIME_INDEX} is outside the available forcing range "
        f"0 to {n_forcing_times - 1}."
    )

if RUNTIME_MODE == "full_forcing":
    RUNTIME_DAYS = max_runtime_days

elif RUNTIME_MODE == "manual":
    RUNTIME_DAYS = float(RUNTIME_DAYS_REQUESTED)

    if RUNTIME_DAYS > max_runtime_days:
        raise ValueError(
            f"Requested runtime is too long.\n"
            f"Requested runtime : {RUNTIME_DAYS:.3f} days\n"
            f"Available runtime : {max_runtime_days:.3f} days\n"
            f"Use RUNTIME_MODE='full_forcing' or reduce RUNTIME_DAYS_REQUESTED."
        )

else:
    raise ValueError(f"Unknown RUNTIME_MODE: {RUNTIME_MODE}")


print("\nRuntime settings")
print(f"  forcing time steps    : {n_forcing_times}")
print(f"  forcing dt            : {TIME_STEP_SECONDS} s")
print(f"  release index         : {RELEASE_TIME_INDEX}")
print(f"  max available runtime : {max_runtime_days:.3f} days")
print(f"  selected runtime      : {RUNTIME_DAYS:.3f} days")


print("\nParticle model convention")
print("  Passive tracer : tau_p_seconds = 0")
print("  MR-SM particle : tau_p_seconds is prescribed directly")
print("  Diameter, buoyancy, viscosity, and drag correction are not used by default.")

In [ ]:
# ============================================================
# 8. PARCELS ADVECTION
# One trajectory file per particle class.
# ============================================================

run_outputs = {}

for spec in particle_specs:
    particle_tag = spec["tag"]
    run_id = f"{particle_tag}_{level_tag}_release_t{RELEASE_TIME_INDEX:04d}"
    out_zarr = RUN_OUT_DIR / f"{run_id}.zarr"
    metadata_json = out_zarr.with_suffix(".json")

    if OVERWRITE_OUTPUT and RUN_ADVECTION and out_zarr.exists():
        if out_zarr.is_dir():
            shutil.rmtree(out_zarr)
        else:
            out_zarr.unlink()
    if OVERWRITE_OUTPUT and RUN_ADVECTION and metadata_json.exists():
        metadata_json.unlink()

    if RUN_ADVECTION:
        config = prun.RunConfig(
            input_nc=str(DATA_FILE),
            output_path=str(out_zarr),
            runtime_days=RUNTIME_DAYS,
            dt_seconds=DT_SECONDS,
            outputdt_seconds=OUTPUTDT_SECONDS,
            time_step_seconds=TIME_STEP_SECONDS,
            release_time_index=RELEASE_TIME_INDEX,
            surface_only=True,
            mesh="flat",
            periodic=PERIODIC,
            level_indices=LEVEL_INDICES,
            release_mode="grid",
            nx=NX,
            ny=NY,
            release_margin_cells=RELEASE_MARGIN_CELLS,
            particle_class=spec["particle_class"],
            particle_tag=spec["tag"],
            particle_label=spec["label"],
            B=spec["B"],
            diameter_m=spec["diameter_m"],
            nu_m2_s=spec["nu_m2_s"],
            f0=spec["f0"],
            drag_correction=spec["drag_correction"],
            C_Rep=spec["C_Rep"],
            Rep_max=spec["Rep_max"],
            flow_timescale_seconds=spec["flow_timescale_seconds"],
            save_metadata_sidecar=True,
        )

        print(f"\nRunning {particle_tag}")
        info = prun.run_parcels_experiment(config)

    else:
        if not out_zarr.exists():
            raise FileNotFoundError(
                f"RUN_ADVECTION=False, but output does not exist:\n{out_zarr}"
            )
        if metadata_json.exists():
            info = prun.load_run_metadata(metadata_json)
        else:
            info = {
                "particle_tag": particle_tag,
                "particle_label": spec["label"],
                "particle_class": spec["particle_class"],
                "output_path": str(out_zarr),
            }

    run_outputs[particle_tag] = {
        "path": str(out_zarr),
        "metadata_path": str(metadata_json),
        "label": info.get("particle_label", spec["label"]),
        "info": info,
        "spec": spec,
    }

    print(f"  output   : {out_zarr}")
    print(f"  metadata : {metadata_json}")


# ============================================================
# 9. SAVE COLLECTION METADATA
# ============================================================

def _json_ready(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {k: _json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_ready(v) for v in obj]
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, float) and not np.isfinite(obj):
        return None
    return obj

config_dict = {
    "created_utc": datetime.utcnow().isoformat() + "Z",
    "case_name": case_name,
    "run_collection_id": run_collection_id,
    "input_nc": str(DATA_FILE),
    "case_results_dir": str(CASE_RESULTS_DIR),
    "trajectory_dir": str(RUN_OUT_DIR),
    "release_time_index": int(RELEASE_TIME_INDEX),
    "release_time_days": float(release_time_days),
    "level_indices": list(LEVEL_INDICES),
    "runtime_mode": RUNTIME_MODE,
    "runtime_days": float(RUNTIME_DAYS),
    "dt_seconds": int(DT_SECONDS),
    "outputdt_seconds": int(OUTPUTDT_SECONDS),
    "time_step_seconds": int(TIME_STEP_SECONDS),
    "nx": int(NX),
    "ny": int(NY),
    "periodic": bool(PERIODIC),
    "release_margin_cells": float(RELEASE_MARGIN_CELLS),
    "flow_timescale_seconds": float(FLOW_TIMESCALE_SECONDS),
    "representative_lat_deg": float(REPRESENTATIVE_LAT_DEG),
    "f0": float(F0),
    "particle_specs": particle_specs,
    "run_outputs": run_outputs,
}

if SAVE_METADATA:
    with open(CONFIG_JSON, "w") as f:
        json.dump(_json_ready(config_dict), f, indent=2)

    print(f"\nSaved collection metadata: {CONFIG_JSON}")

print("\nDone. Use this in the analysis notebook:")
print(f"RUN_COLLECTION_ID = {run_collection_id!r}")
